# Flash Attention v2 — Triton Kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qooba/bielik-anatomy-triton/blob/main/notebooks/ep05-flash-attention.ipynb)

Part of the [Bielik Anatomy](https://github.com/qooba/bielik-anatomy-triton) series — Episode 5: Flash Attention v2. Full write-up: [ep05-flash-attention.md](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep05-flash-attention.md).

Standard attention materializes an O(N^2) score matrix in HBM. Flash Attention tiles the computation and uses online softmax to keep everything in fast on-chip SRAM. This notebook runs `kernels/attention/flash_attention_simple.py` against PyTorch's `scaled_dot_product_attention`.

This notebook lets you run this kernel's correctness check and benchmark on a free Colab GPU — no local NVIDIA hardware required.

**Runtime:** select a GPU runtime before running: `Runtime -> Change runtime type -> T4 GPU` (free tier works fine).

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_DIR = "bielik-anatomy-triton"
if not os.path.isdir(REPO_DIR):
    !git clone --quiet https://github.com/qooba/bielik-anatomy-triton {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 1. Correctness check

Compare the Triton kernel output against a plain PyTorch reference implementation.

In [ ]:
import torch
import torch.nn.functional as F
from kernels.attention.flash_attention_simple import flash_attention_forward

torch.manual_seed(0)
batch, num_heads, seq_len, head_dim = 2, 8, 256, 64
q = torch.randn(batch, num_heads, seq_len, head_dim, device="cuda", dtype=torch.float16)
k = torch.randn(batch, num_heads, seq_len, head_dim, device="cuda", dtype=torch.float16)
v = torch.randn(batch, num_heads, seq_len, head_dim, device="cuda", dtype=torch.float16)

out_triton = flash_attention_forward(q, k, v, causal=True)
out_ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)

max_diff = (out_triton - out_ref).abs().max().item()
print(f"max abs diff vs PyTorch SDPA: {max_diff:.6f}")
assert torch.allclose(out_triton, out_ref, atol=2e-2, rtol=2e-2)
print("Correctness check passed.")

## 2. Benchmark

Runs the same sweep as `make benchmark-flash-attention` and regenerates the plots used in the episode doc. Takes roughly 1-3 minutes on a T4.

In [ ]:
!PYTHONPATH=. python benchmarks/attention/benchmark_flash_attention.py --save-plots --plot-dir=docs/plots/attention/

## 3. Results

In [ ]:
from IPython.display import Image, display

plot_dir = "docs/plots/attention/"
for fname in [
    "flash-attention-tflops-vs-heads.png",
    "flash-attention-tflops-vs-seq-len.png",
]:
    display(Image(filename=f"{plot_dir}{fname}"))

---
[Full episode write-up](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep05-flash-attention.md) · [Series overview](https://github.com/qooba/bielik-anatomy-triton/blob/main/README.md)